# Capítulo 2 — Probabilidade e Distribuições

Notebook com o **código** deste capítulo, para o Google Colab. Cada trecho vem precedido de uma explicação curta; o texto completo está no site do livro.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [ ]:
# Setup (rode uma vez).
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 2.1 — O que é Probabilidade

Simula milhares de lançamentos de uma moeda justa e acompanha a proporção acumulada de caras: com poucos lançamentos ela oscila bastante, mas conforme o número de tentativas cresce, se aproxima de 0,5 — a visão frequentista de probabilidade como frequência no longo prazo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
rng = np.random.default_rng(42)
lancamentos = rng.integers(0, 2, 10000)          # 0 ou 1, moeda justa
proporcao = np.cumsum(lancamentos) / np.arange(1, 10001)

for n in [10, 100, 1000, 10000]:
    print(f"Após {n:>5} lançamentos: proporção de caras = {num(proporcao[n-1], 3)}")

Plota essa proporção acumulada num eixo horizontal em escala logarítmica, para que a oscilação inicial (poucos lançamentos) e a estabilização final (milhares de lançamentos) fiquem igualmente visíveis.

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.arange(1, 10001), proporcao, color="#2c3e50", linewidth=1)
ax.axhline(0.5, color="#c0392b", linestyle="--", linewidth=2)
ax.set_xscale("log")
ax.set_xlabel("Número de lançamentos (escala log)")
ax.set_ylabel("Proporção de caras")
plt.tight_layout()
plt.show()

## 2.2 — Regras de Probabilidade

Calcula a probabilidade do complementar: se 2% das requisições a um serviço falham, as outras 98% não falham — as duas probabilidades sempre somam 1.

In [ ]:
from formato import num

p_falha = 0.02
p_sucesso = 1 - p_falha
print(f"P(requisição falhar)     : {num(p_falha, 2)}")
print(f"P(requisição não falhar) : {num(p_sucesso, 2)}")

Multiplica probabilidades de dois serviços independentes, cada um 99% disponível, para achar a chance de ambos estarem no ar; usa o complementar para achar a chance de pelo menos um cair.

In [ ]:
from formato import num
p = 0.99
print(f"Ambos os serviços no ar:      {num(p**2, 4)}")
print(f"Pelo menos um fora do ar:     {num(1 - p**2, 4)}")

Constrói a distribuição de probabilidade completa do número de caras em dois lançamentos independentes de uma moeda, combinando a regra da multiplicação (eventos independentes) com a regra da adição (eventos mutuamente exclusivos).

In [ ]:
from formato import num

p_caras = {
    0: 0.5**2,           # coroa-coroa
    1: 2 * 0.5**2,       # cara-coroa OU coroa-cara
    2: 0.5**2,           # cara-cara
}
for k, prob in p_caras.items():
    print(f"P(X = {k}) = {num(prob, 2)}")

## 2.3 — Probabilidade Condicional e Bayes

In [ ]:
import numpy as np
from formato import num

Aplica o Teorema de Bayes com três cenários (candidatos bons, médios e fracos): a partir do prior de cada categoria e da chance de aprovação em cada uma, calcula o posterior de um candidato aprovado ser fraco.

In [ ]:
prior = {"bom": 0.25, "médio": 0.50, "fraco": 0.25}
aprovado_dado = {"bom": 0.80, "médio": 0.50, "fraco": 0.20}
evidencia = sum(prior[c] * aprovado_dado[c] for c in prior)
posterior_fraco = prior["fraco"] * aprovado_dado["fraco"] / evidencia
print(f"P(candidato fraco | foi aprovado) = {num(posterior_fraco, 2)}")

Aplica Bayes à armadilha da taxa-base: com um evento raro (1% de prevalência), mesmo um alarme com 80% de sensibilidade acerta bem menos do que a intuição sugere quando dispara.

In [ ]:
prevalencia, sensibilidade, falso_positivo = 0.01, 0.80, 0.096
posterior = (prevalencia * sensibilidade) / (
    prevalencia * sensibilidade + (1 - prevalencia) * falso_positivo)
print(f"P(evento real | alarme disparou) = {num(posterior * 100, 1)}%")

Confirma o resultado anterior por simulação de Monte Carlo: gera um milhão de casos, cada um com sua chance de ter o evento e de disparar o alarme, e conta a fração de alarmes que correspondem a um evento real.

In [ ]:
rng = np.random.default_rng(42)
n = 1_000_000
tem_evento = rng.random(n) < prevalencia
alarme = np.where(tem_evento, rng.random(n) < sensibilidade, rng.random(n) < falso_positivo)
positivos = int(alarme.sum())
reais = int((alarme & tem_evento).sum())
print(f"De {num(positivos, 0)} alarmes, {num(reais, 0)} eram evento real: {num(100 * reais / positivos, 1)}%")

## 2.4 — Contagem: Permutação e Combinação

In [ ]:
from math import factorial, perm, comb
from formato import num

Compara fatorial, permutação e combinação num mesmo exemplo: quantas ordens 5 corredores podem cruzar a linha de chegada, quantos pódios de 3 lugares existem, e quantos subconjuntos de 4 itens cabem em 20, sem se importar com a ordem.

In [ ]:
print(f"5! = {factorial(5)}")                    # permutações de 5 elementos
print(f"P(5,3) = {perm(5, 3)}")                  # pódios ordenados de 3 entre 5
print(f"C(20,4) = {num(comb(20, 4), 0)}")        # amostras (sem ordem) de 4 entre 20

Usa combinações para calcular a probabilidade de sortear exatamente 2 peças defeituosas entre 4 retiradas de um lote de 20 com 5 defeituosas — a mesma estrutura que a seção 2.6 generaliza como distribuição hipergeométrica.

In [ ]:
p = comb(5, 2) * comb(15, 2) / comb(20, 4)
print(f"P(2 defeituosas entre as 4 sorteadas) = {num(p, 3)}")

## 2.5 — Distribuição Binomial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Desenha a distribuição binomial do número de sucessos em 20 tentativas com p=0,1: um gráfico de barras, porque o número de sucessos só assume valores inteiros, com pico perto da média np=2.

In [ ]:
k = np.arange(0, 11)
fig, ax = plt.subplots()
ax.bar(k, stats.binom.pmf(k, 20, 0.1), color="#b0c4d8", edgecolor="white")
ax.set_xlabel("Número de sucessos")
ax.set_ylabel("Probabilidade")
plt.tight_layout()
plt.show()

Compara a probabilidade de exatamente 2 sucessos (`pmf`) com a de até 2 sucessos (`cdf`) em 5 tentativas com p=0,1: a primeira é rara, a segunda é quase certa, a assinatura de eventos pouco frequentes.

In [ ]:
print(f"P(exatamente 2 sucessos em 5, p=0,1): {num(stats.binom.pmf(2, 5, 0.1), 4)}")
print(f"P(até 2 sucessos em 5, p=0,1):        {num(stats.binom.cdf(2, 5, 0.1), 4)}")

## 2.6 — Distribuição Hipergeométrica

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import hypergeom, binom
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Calcula, para um lote de 100 casos de teste com 10 com bug, a chance de uma amostra de 5 sair totalmente limpa e a chance de pegar ao menos um bug — sem reposição, a premissa que separa a hipergeométrica da binomial.

In [ ]:
# scipy: hypergeom(M, n, N) = (população, sucessos na população, tamanho da amostra)
# livro : N=100 (lote), r=10 (defeituosas), n=5 (amostra sem reposição)
lote = hypergeom(100, 10, 5)
print(f"P(0 defeituosas na amostra) = {num(float(lote.pmf(0)), 3)}")
print(f"P(ao menos 1 defeituosa)    = {num(float(1 - lote.cdf(0)), 3)}")

Compara a hipergeométrica (sem reposição) com a binomial equivalente (com reposição, p=r/N): como a amostra é só 5% do lote, as duas quase coincidem — a regra dos 10%.

In [ ]:
print(f"Hipergeométrica (sem reposição): {num(float(lote.pmf(0)), 3)}")
print(f"Binomial (com reposição, p=0,1): {num(float(binom.pmf(0, 5, 0.1)), 3)}")

Desenha a distribuição hipergeométrica completa do número de defeituosas numa amostra de 5 tirada do lote de 100 com 10 defeituosas: o suporte vai só de 0 a 5, e a maior parte da massa fica em 0.

In [ ]:
k = np.arange(0, 6)
fig, ax = plt.subplots()
ax.bar(k, lote.pmf(k), color="#b0c4d8", edgecolor="white")
ax.set_xlabel("Número de defeituosas na amostra")
ax.set_ylabel("Probabilidade")
plt.tight_layout()
plt.show()

## 2.7 — Distribuição Normal

Importa as bibliotecas e carrega a renda anual de 50.000 solicitantes de empréstimo (em dólares), usada nesta seção para testar visualmente se um dado real segue a normal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

Desenha a curva normal com as faixas de 1, 2 e 3 desvios-padrão marcadas — a regra 68-95-99,7, que diz quanta massa fica dentro de cada faixa.

In [ ]:
x = np.linspace(-4, 4, 200)
fig, ax = plt.subplots()
ax.plot(x, stats.norm.pdf(x), color="#2c3e50", linewidth=2)
for k, cor in [(1, "#27ae60"), (2, "#e67e22"), (3, "#c0392b")]:
    ax.axvline(k, color=cor, linestyle="--", alpha=0.6)
    ax.axvline(-k, color=cor, linestyle="--", alpha=0.6)
ax.set_xlabel("Desvios em relação à média")
ax.set_ylabel("Densidade")
plt.tight_layout()
plt.show()

Desenha o QQ-plot da renda contra a normal: o corpo dos pontos segue a reta, mas a ponta direita sobe para longe dela, revelando a cauda longa da renda.

In [ ]:
fig, ax = plt.subplots()
stats.probplot(renda, dist="norm", plot=ax)
ax.set_title("")
ax.set_xlabel("Quantis teóricos (normal)")
ax.set_ylabel("Quantis da renda")
plt.tight_layout()
plt.show()

Mede a assimetria da renda: um valor positivo confirma a cauda à direita já vista no QQ-plot, contra o zero que uma normal teria.

In [ ]:
print(f"Assimetria da renda: {num(stats.skew(renda), 2)}")

## 2.8 — Distribuições de Cauda Longa

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)

Gera uma amostra genuinamente de cauda gorda (uma t de Student com poucos graus de liberdade) e mede sua curtose: bem acima de zero, o valor de uma normal, confirmando caudas muito mais pesadas.

In [ ]:
amostra = stats.t.rvs(df=3, size=1000, random_state=42)
print(f"Curtose da amostra:      {num(stats.kurtosis(amostra), 1)}")
print(f"Curtose de uma normal:   0")

Desenha o QQ-plot dessa amostra contra a normal: a curva sai em S, subindo à direita e descendo à esquerda — a marca de uma distribuição simétrica que infla os dois extremos ao mesmo tempo, diferente da cauda assimétrica da renda.

In [ ]:
fig, ax = plt.subplots()
stats.probplot(amostra, dist="norm", plot=ax)
ax.set_title("")
ax.set_xlabel("Quantis teóricos (normal)")
ax.set_ylabel("Quantis da amostra")
plt.tight_layout()
plt.show()